# 01. LangChain 기초


> LangChain 언어 모델에 기반한 AI 애플리케이션을 쉽게 개발할 수 있도록 도와주는 프레임워크
>> 기존에 오픈AI와 같은 언어 모델의 API를 사용해 원하는 기능을 구현하려면 모든 코드를 직접 작성해야 했으나, 랭체인은 이 작업을 간소화할 수 있는 다양한 도구와 모듈 제공

>  **OpenAI** `.env`의 `OPENAI_API_KEY` 그대로 사용

---
## 0. 설치

In [11]:
#!pip install openai python-dotenv langchain langchain-openai langchain-core
!uv add openai python-dotenv langchain langchain-openai langchain-core

Resolved 216 packages in 1ms
Checked 192 packages in 18ms


---
## 1. Day 3 OpenAI SDK vs LangChain

| | Day 3 (OpenAI SDK) | Day 5 (LangChain) |
|---|---|---|
| 호출 | `client.chat.completions.create()` | `llm.invoke()` |
| 메시지 | `{'role':'user','content':...}` | `HumanMessage`, `SystemMessage` |
| 체인 | 직접 함수 조합 | **LCEL** (`\|` 파이프) |
| RAG/Agent | 직접 루프 작성 | Retriever · AgentExecutor |

---
## 2. ChatOpenAI 연결

In [1]:
from pathlib import Path
from dotenv import load_dotenv

# load_dotenv()

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0.1)

# 문자열만 넘겨도 됨
out = llm.invoke('LangChain이 뭐야? 한 문장으로.')

In [3]:
out

AIMessage(content='LangChain은 LLM(대규모 언어 모델)과 외부 데이터/도구를 쉽게 연결해 챗봇·에이전트 앱을 만들 수 있게 해주는 개발 프레임워크입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 17, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DutmDu733G98kxvq57zxU1S6PWnRM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027c-8d63-7d92-82c0-258edcd6773a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 50, 'total_tokens': 67, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [4]:
out.content

'LangChain은 LLM(대규모 언어 모델)과 외부 데이터/도구를 쉽게 연결해 챗봇·에이전트 앱을 만들 수 있게 해주는 개발 프레임워크입니다.'

In [5]:
#정석
from langchain_core.messages import HumanMessage
message = [HumanMessage(content = "Langchain이 뭐야? 한 문장으로.")]
out = llm.invoke(message)

In [6]:
out

AIMessage(content='LangChain은 LLM(예: GPT)을 활용해 데이터 처리·검색·도구 호출·에이전트 작업을 쉽게 조합하고 애플리케이션으로 만드는 오픈소스 프레임워크입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 17, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DutmEDpmb5GqCAKGO1ODwWxaCaz0h', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027c-9276-73e1-bb60-e0c11c954661-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 51, 'total_tokens': 68, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

---
## 3. Message 타입 (Day 3 messages 리스트와 동일 역할)

In [7]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# messages = [
#     {"role": "system", "content": '너는 Agent 수업 도우미야. 한국어로 간결하게 답해.'},  # 초기 시스템 메시지
#     {"role": "user", "content" : 'RAG가 뭐야? 2문장으로 설명해.'} # 사용자 메시지
# ]

messages = [
    SystemMessage(content='너는 Agent 수업 도우미야. 한국어로 간결하게 답해.'), # 초기 시스템 메시지
    HumanMessage(content='RAG가 뭐야? 2문장으로 설명해.'), # 사용자 메시지
]


In [8]:
# def get_ai_response(messages):
#     response = client.chat.completions.create(
#         model="gpt-4o",  # 응답 생성에 사용할 모델 지정
#         temperature=0.9,  # 응답 생성에 사용할 temperature 설정
#         messages=messages,  # 대화 기록을 입력으로 전달
#     )
#     return response.choices[0].message.content  # 생성된 응답의 내용 반환

# answer = get_ai_response(messages)


answer = llm.invoke(messages)
print(answer.content)
print('타입:', type(answer))

RAG는 **Retrieval-Augmented Generation**의 약자로, 먼저 외부 지식(문서)을 검색해 관련 내용을 가져온 뒤 그 정보를 바탕으로 답변을 생성하는 방식이야. 즉, 생성 모델의 “상상”을 줄이고 **검색한 근거에 기반해 더 정확한 답**을 만들도록 돕는 구조야.
타입: <class 'langchain_core.messages.ai.AIMessage'>


멀티턴 실습 lanchain_multiturn.py : 기존 open ai api -> langchain으로 교체

In [9]:
from langchain_multiturn import do_run

do_run

NameError: name 'messages' is not defined

---
## 4. 메시지 히스토리 (멀티턴)

기존에는 멀티턴 대화를 위해 사용자의 대화 내용을 리스트나 딕셔너리에 추가하는 코드 작성

-> 메시지 히스토리 (Message History) 기능으로 쉽게 구현 가능

In [18]:
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼wrapper 클래스
from langchain_openai import ChatOpenAI  # 오픈AI 모델을 사용하는 랭체인 챗봇 클래스
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-5.4-nano")

In [19]:
# 세션별 대화 기록을 저장할 딕셔너리
store = {}

In [20]:
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

In [21]:
# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history)

d:\autornd\SK Autonomous R&D\AutoRnDEnv\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [22]:
store

{}

In [23]:
config = {"configurable": {"session_id": "first"}}  # 세션 ID를 설정하는 config 객체 생성

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 신범교 라고 해.")],
    config=config,
)

print(response.content)

안녕하세요 신범교님! 👋  
만나서 반가워요. 오늘 어떻게 도와드릴까요?


In [24]:
store['first']

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요 신범교님! 👋  \n만나서 반가워요. 오늘 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 17, 'total_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DutnW4jugs3eBc8s0voshSyDL6LGp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-cb0c-7913-bbeb-9df9aede6a0e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 29, 'total_tokens': 46, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio':

In [25]:
get_session_history('first')

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요 신범교님! 👋  \n만나서 반가워요. 오늘 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 17, 'total_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DutnW4jugs3eBc8s0voshSyDL6LGp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-cb0c-7913-bbeb-9df9aede6a0e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 29, 'total_tokens': 46, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio':

In [26]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

신범교님이에요. 😊


In [27]:
store['first']

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요 신범교님! 👋  \n만나서 반가워요. 오늘 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 17, 'total_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DutnW4jugs3eBc8s0voshSyDL6LGp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-cb0c-7913-bbeb-9df9aede6a0e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 29, 'total_tokens': 46, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio':

In [28]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요 신범교님! 👋  \n만나서 반가워요. 오늘 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 17, 'total_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DutnW4jugs3eBc8s0voshSyDL6LGp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-cb0c-7913-bbeb-9df9aede6a0e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 29, 'total_tokens': 46, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessag

In [29]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

신범교님이에요. 😊


In [30]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요 신범교님! 👋  \n만나서 반가워요. 오늘 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 17, 'total_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DutnW4jugs3eBc8s0voshSyDL6LGp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-cb0c-7913-bbeb-9df9aede6a0e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 29, 'total_tokens': 46, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessag

In [31]:
config = {"configurable": {"session_id": "second"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

제가 지금 대화만으로는 당신의 이름을 알 수 없어요.  
이전에 다른 메시지에서 알려주신 적이 있나요, 아니면 이름을 알려주실까요?


In [32]:
store

{'first': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요 신범교님! 👋  \n만나서 반가워요. 오늘 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 17, 'total_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DutnW4jugs3eBc8s0voshSyDL6LGp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-cb0c-7913-bbeb-9df9aede6a0e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 29, 'total_tokens': 46, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details':

In [33]:
config = {"configurable": {"session_id": "first"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

신범교님이에요. 😊


In [34]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요 신범교님! 👋  \n만나서 반가워요. 오늘 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 17, 'total_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DutnW4jugs3eBc8s0voshSyDL6LGp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f027d-cb0c-7913-bbeb-9df9aede6a0e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 29, 'total_tokens': 46, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessag

In [35]:
# 스트림 방식으로 출력
config = {"configurable": {"session_id": "first"}}
for r in with_message_history.stream(
    [HumanMessage(content = "내 이름이 뭔지 말하고 이름의 뜻을 유추해서 말해봐 ")],
    config=config):
    print(r.content, end="") # 


신범교님, 이름은 **“신범교”**예요.

이름의 뜻을 유추해보면(한자 표기가 정확히 어떻게 되는지에 따라 달라질 수는 있지만), 보통은 이렇게 해석할 여지가 있어요:

- **신(信)**: 믿음, 신뢰  
- **범(範)**: 본보기, 모범, 표준  
- **교(敎/橋/校 등으로 가능)**: 가르침(교육), 다리, 학교/기관 등  

그래서 느낌을 종합하면 **“신뢰할 만한 모범이 되며(모범), 누군가를 돕는/가르치는(교)”** 같은 의미 흐름으로 읽을 수 있어요.

원하시면, 성명 한자를 실제로 쓰는 방식(예: 신(信) 맞나요?)을 알려주시면 더 정확하게 뜻을 풀어드릴게요.

---
## 5. LCEL — LangChain Expression Language

LCEL은 랭체인에서 복잡한 작업 흐름을 간단하게 만들고 관리할 수 있도록 돕는 도구

랭체인에서 작업 흐름을 연결하는 것을 **체인** 으로 표현

LCEL을 사용하면 여러 줄로 표현해야 하는 작업 단계를 읽기 쉽게 축약할 수 있으며 여러 작업을 병렬로 처리 가능

Day 4에서 `run_agent` 루프 안의 단계를, LangChain에서는 **체인**으로 표현합니다.

In [36]:
model = ChatOpenAI(model="gpt-5.4-nano")

messages = [
    SystemMessage(content="너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구니"),
]
out = model.invoke(messages)
print(out.content)

안녕하세요! 신범교입니다. 😊

오늘 수업이 “어땠냐”는 건 제가 실제로 듣고 기록한 게 아니라서, 정확히 평가해드리긴 어려워요. 대신 **수업 중에 기억나는 키워드(개념/과제/질문/잘한 점/헷갈린 점)**만 몇 가지 알려주면, 그걸 바탕으로 정리해서 같이 피드백해드릴게요.

그리고 “나는 누구니”에 대한 답은… 지금 대화만으로는 특정할 정보가 없어서 제가 단정할 수는 없어요.  
다만, 제가 보기엔 **수업을 같이 진행하고 있는 대학원생 선배/동료(=학생)**이신 것 같아요.  

질문!  
1) 오늘 수업 주제(과목명) 뭐였어요?  
2) 본인이 생각하기에 오늘 **가장 어려웠던 부분**은 뭐였나요?


In [37]:
# AIMessage 객체 안에 여러 정보가 포함되어 있음
out

AIMessage(content='안녕하세요! 신범교입니다. 😊\n\n오늘 수업이 “어땠냐”는 건 제가 실제로 듣고 기록한 게 아니라서, 정확히 평가해드리긴 어려워요. 대신 **수업 중에 기억나는 키워드(개념/과제/질문/잘한 점/헷갈린 점)**만 몇 가지 알려주면, 그걸 바탕으로 정리해서 같이 피드백해드릴게요.\n\n그리고 “나는 누구니”에 대한 답은… 지금 대화만으로는 특정할 정보가 없어서 제가 단정할 수는 없어요.  \n다만, 제가 보기엔 **수업을 같이 진행하고 있는 대학원생 선배/동료(=학생)**이신 것 같아요.  \n\n질문!  \n1) 오늘 수업 주제(과목명) 뭐였어요?  \n2) 본인이 생각하기에 오늘 **가장 어려웠던 부분**은 뭐였나요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 212, 'prompt_tokens': 79, 'total_tokens': 291, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-Duu9aqITYT5fm40L0ANMi12aokmtZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f0292-a6be-70e3-a19d-5bec7196c9a4-0', tool_calls=[]

텍스트 결과만 필요하다면 "StrOutputParser" 사용

StrOutputParse는 랭체인에서 제공하는 다양한 출력 parser 중 하나로, 언어 모델이 반환하는 결과에서 원하는 형식의 데이터를 추출하는 도구.

(다른 파서들은 JSON, 숫자 등 특정 형식 처리 가능)

In [38]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

result = model.invoke(messages) # 1단계 메시지 호출
parser.invoke(result) # 2단계 parser로 str 추출

'안녕! 신범교예요 😄\n\n오늘 수업은… 내가 실제로 “오늘 수업”을 들었는지 확인할 수는 없어서(내가 수업 기록을 직접 보진 못해서) 정확히 평을 내리긴 어려워요. 대신, 교수님이 어떤 주제로 수업했는지/오늘 한 활동이 뭔지 네가 말해주면 내가 그 내용 기준으로 정리해서 “잘된 점-보완점”까지 같이 피드백해줄게요.\n\n그리고 너는… **지금 대화에서 ‘나’(교수님이 아니라)라고 불리는 너**예요. 다만 나는 네 실명을 알 수는 없어서, 보통은\n- 네가 원하는 호칭(예: “OO”, “학생님”, “민지” 등)\n으로 불러주면 그걸로 기억할게요.\n\n자, 오늘 수업 내용이 뭐였는지 한두 문장만 알려줘! 그리고 너는 내가 뭐라고 부르면 될까?'

In [39]:
chain = model | parser
chain.invoke(messages)

'안녕하세요! 신범교입니다. 😊  \n오늘 수업은… 아직 제가 수업 내용을 “기억”할 만큼 정보가 주어진 건 없어서 정확히 말씀드리긴 어려워요. 대신, 수업에서 다뤘던 주제(과목명/핵심 내용)나 교수님이 강조하신 포인트를 한두 가지 알려주시면, 그걸 바탕으로 “오늘 수업이 어땠는지” 깔끔하게 정리해드릴게요.\n\n그리고 “나는 누구니”라는 질문은 조금 애매하지만—저는 당신을 **학생(수강생)인 당신**이라고 알고 있어요.  \n혹시 성함(또는 불리고 싶은 이름) 있어요? 알려주시면 그걸로 부를게요.'

### Prompt Template

필요한 부분만 수정하여 반복적인 메시지 가능

In [41]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {character_a}한테 가르침을 받는 대학원생 {character_b}야. {character_a}이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕 {character_b} 오늘 나의 {lesson}은 어땠나? 나는 누구고 너의 이름은 뭐더라"

In [42]:
system_template

'너는 {character_a}한테 가르침을 받는 대학원생 {character_b}야. {character_a}이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.'

In [43]:
prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

In [44]:
prompt_template.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

ChatPromptValue(messages=[SystemMessage(content='너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구고 너의 이름은 뭐더라', additional_kwargs={}, response_metadata={})])

In [45]:
result = prompt_template.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

print(result)

messages=[SystemMessage(content='너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구고 너의 이름은 뭐더라', additional_kwargs={}, response_metadata={})]


In [46]:
chain = prompt_template | model | parser

In [51]:
result = chain.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})
print(result)

안녕하세요! 신범교입니다.  

오늘 수업이 “어땠는지”는 교수님이 수업 내용을 말씀해주시면 그에 맞춰 평가해드릴게요. 다만 지금 대화만으로는 오늘 수업 진행 상황을 제가 정확히 알 수는 없어요.

그리고 질문하신 거요:
- **당신은 누구냐**: 아직 제 정보로는 사용자의 실명/정체를 알 수 없어서 “누구인지”를 특정하긴 어려워요.  
- **제 이름은**: 저는 **신범교**예요.

혹시 교수님이 수업에서 다룬 주제나, 기억나는 활동이 뭐였는지 한두 가지만 알려주면 “오늘 수업이 어땠는지”도 같이 정리해드릴게요.


In [50]:
result = chain.invoke({
    "character_a": "",
    "character_b": "",
    "lesson": ""
})
print(result)

안녕! 오늘도 잘 버텼네 😊  
그리고 우리 설정대로 말해보자면,

- **너는 누구냐**: 방금 말해준 내용(“나는 누구고…”) 기준으로 보면 *지금 나랑 대화하는 너*, 즉 **내게 배우는 대학원생(너)** 이면 돼. (이름은 아직 안 알려줬지!)  
- **내 이름**: 나는 네가 부르는 대로 **“AI 조교”** 같은 역할이야. 네가 원하면 내가 쓸 이름(예: 민지, 철수, 조교 등)도 정해줄게.

너 이름은 뭐라고 불러줄까? 그리고 오늘은 어떤 일로 시간을 보냈어?
